# Otimização Avançada — busca de hiperparâmetros por NDCG@7

Um sistema de recomendação **não se avalia por MSE**. O objetivo é *ordenar* o
catálogo, então a métrica de seleção é o **NDCG@7 de validação** (leave-last-out).

Neste notebook fazemos uma busca **enxuta** de hiperparâmetros no espaço que mais
importa para generalização em bases esparsas: **capacidade do embedding** e
**regularização** (`l2` + `dropout`). Cada configuração é treinada com early stopping
pelo próprio NDCG@7 de validação (`RankingCheckpoint`).

## 1. Configuração do ambiente

In [ ]:
import sys
from pathlib import Path

# Localiza a raiz do projeto (onde está o pyproject.toml) para importar `src.*`.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

import keras
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder

from src.config.model_utils import (
    TOP_K,
    RankingCheckpoint,
    build_item_catalog,
    build_training_pairs,
    categories_for_products,
    create_ranking_model,
    evaluate_ranking,
    train_ranking_model,
    train_validation_test_split,
)
from src.config.paths import DADOS_TRATADOS

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
print("Projeto:", ROOT)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)

## 2. Dados, split e pares de treino

O treino aprende a **ordenar positivos acima de negativos** (amostragem de negativos),
o que é a formulação correta para ranking implícito.

In [ ]:
dados = pd.read_parquet(DADOS_TRATADOS)
encoders = {
    "customer": LabelEncoder().fit(dados["Customer Name"]),
    "product": LabelEncoder().fit(dados["Product ID"]),
    "category": LabelEncoder().fit(dados["Category"]),
    "subcategory": LabelEncoder().fit(dados["Sub-Category"]),
}
num_produtos = len(encoders["product"].classes_)
num_categorias = len(encoders["category"].classes_)
num_subcategorias = len(encoders["subcategory"].classes_)
num_clientes = len(encoders["customer"].classes_)

catalogo = build_item_catalog(
    dados, encoders["product"], encoders["category"], encoders["subcategory"]
)
train_pairs, val_pairs, test_pairs = train_validation_test_split(
    dados, encoders["customer"], encoders["product"]
)

# Positivos + 8 negativos por interação (perda de ranking binária).
train_customers, train_products, train_labels = build_training_pairs(
    train_pairs, num_produtos, n_negatives=8, seed=42
)
train_categorias, train_subcategorias = categories_for_products(train_products, catalogo)

print("Dimensões:", {"clientes": num_clientes, "produtos": num_produtos,
                     "categorias": num_categorias, "subcategorias": num_subcategorias})
print("Amostras de treino (positivos+negativos):", len(train_labels))

## 3. Busca de hiperparâmetros

Três configurações representativas: um modelo pequeno, o default de produção e um
modelo mais largo/regularizado. A seleção é pelo **melhor NDCG@7 de validação**.

In [ ]:
configuracoes = [
    {"nome": "compacto", "embedding_dim": 16, "dropout_rate": 0.3, "l2_strength": 1e-3, "learning_rate": 1e-3},
    {"nome": "produção", "embedding_dim": 32, "dropout_rate": 0.3, "l2_strength": 1e-3, "learning_rate": 1e-3},
    {"nome": "regularizado", "embedding_dim": 64, "dropout_rate": 0.5, "l2_strength": 1e-2, "learning_rate": 5e-4},
]

resultados = []
for cfg in configuracoes:
    print("=" * 70)
    print(f"Configuração: {cfg['nome']} | {cfg}")
    modelo = create_ranking_model(
        num_customers=num_clientes,
        num_products=num_produtos,
        num_categories=num_categorias,
        num_subcategories=num_subcategorias,
        embedding_dim=cfg["embedding_dim"],
        l2_strength=cfg["l2_strength"],
        dropout_rate=cfg["dropout_rate"],
        learning_rate=cfg["learning_rate"],
    )
    checkpoint = RankingCheckpoint(val_pairs, train_pairs, catalogo, k=TOP_K, patience=3, verbose=0)
    train_ranking_model(
        modelo,
        train_customers=train_customers,
        train_products=train_products,
        categories=train_categorias,
        subcategories=train_subcategorias,
        labels=train_labels,
        callbacks=[checkpoint],
        epochs=12,
        batch_size=256,
        verbose=0,
    )
    ndcg_val = max(checkpoint.history.values())
    ndcg_teste = evaluate_ranking(modelo, test_pairs, train_pairs, catalogo, k=TOP_K)[f"ndcg@{TOP_K}"]
    print(f"  melhores épocas: {checkpoint.best_epoch} | NDCG@{TOP_K} val: {ndcg_val:.5f} | teste: {ndcg_teste:.5f}")
    resultados.append({
        "Configuração": cfg["nome"],
        "dim": cfg["embedding_dim"],
        "dropout": cfg["dropout_rate"],
        "l2": cfg["l2_strength"],
        "lr": cfg["learning_rate"],
        f"NDCG@{TOP_K} val": ndcg_val,
        f"NDCG@{TOP_K} teste": ndcg_teste,
    })

## 4. Resultados da busca

In [ ]:
tabela = pd.DataFrame(resultados).set_index("Configuração")
melhor = tabela[f"NDCG@{TOP_K} val"].idxmax()
print(f"Melhor configuração por NDCG@{TOP_K} de validação: {melhor}")
tabela.round(5)

## 5. Conclusão

- A seleção de modelo usa a **métrica-alvo** (NDCG@7 de validação), não um proxy.
- Em bases esparsas, **regularização forte** tende a superar embeddings grandes, que
  decoram o treino sem generalizar.
- A busca é deliberadamente enxuta: para produção, trocar por um *sweep* (Optuna/
  `keras-tuner`) com a mesma métrica de seleção. O `RankingCheckpoint` já é o ponto de
  integração.

O baseline de popularidade desta base é forte; por isso a decisão final de peso entre
popularidade e personalização é feita **fora** daqui, na seleção de `alpha` (ver
notebook 07).